# Higgs Boson Classification with XGBoost in Jupyter Notebook

**Question:** How well do the basic kinematic/tagging features distinguish signal from background compared with the derived mass features?

Complete the entire exercise in this browser-based Jupyter Notebook on an allocated HPC **compute node**. First explore the observables, then train a model with the **21 basic features**. Continue in the **same notebook and kernel** to train a second model with the **seven physics-derived features**. Finish by comparing both models on the same test events.

Run code cells from top to bottom. Both models use identical settings and event splits. A GPU is not required. These are simulated events.

the notebook saves the outputs under `results/higgs_notebook`.

# Stage 1: Explore the data and train with 21 basic features

Complete the setup and Exercise 1, then train and evaluate the basic-feature model. Keep the Jupyter session running for Stage 2.

## 1. Prepare software and data

Use the workshop environment containing `xgboost`, `scikit-learn`, `pandas`, `numpy`, `matplotlib`, `notebook`, and `ipykernel`. Run the following commands in the **compute node**, before starting the notebook:

```bash
vpkg_require miniforge
conda activate "$WORKDIR_USER/sw/ide_workshop"
```

Create the environment using the setup instructions in the slides. Start Jupyter from the active environment on your allocated compute node, then open this notebook in your local web browser through the SSH tunnel. Select the kernel that uses this project environment. The next cell prints its Python executable.

This repository includes **higgs_subset.csv.gz**, a losslessly compressed copy of the workshop CSV. Keep it beside this notebook. Pandas reads it directly without extraction. An existing uncompressed **higgs_subset.csv** also works. Change `DATA_PATH` below if your data is elsewhere.

The supplied subset contains the first **550,000 simulated events** from the [UCI HIGGS dataset](https://archive.ics.uci.edu/dataset/280/higgs), with a header added. This is 5% of the original 11 million events. All 28 input features are retained. It is an introductory subset, not a random sample of the entire dataset. Citation and license appear at the end.

The fixed 60%/20%/20% split gives **330,000 training**, **110,000 validation**, and **110,000 test** events.

In [ ]:
import os
import sys
import json
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

plt.rcParams.update({'figure.figsize': (7, 4), 'font.size': 11})
print('Python:', sys.executable)
print('XGBoost:', xgb.__version__)
print('Working directory:', Path.cwd())

## 2. Set the data path and CPU threads

Change `DATA_PATH` if the CSV is in a different directory, for example `Path('/shared/workshop/higgs_subset.csv')`. Relative paths refer to the working directory printed above.

`N_THREADS` uses the Slurm CPU allocation if available, otherwise one thread. Keep it within your allocated CPU count. Results are saved to `results/higgs_notebook` under the working directory. Use this same directory for both model stages.

In [ ]:
# Prefer an existing CSV, otherwise read the repository's compressed dataset.
DATA_PATH = Path('higgs_subset.csv')
if not DATA_PATH.is_file():
    DATA_PATH = Path('higgs_subset.csv.gz')
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f'CSV not found: {DATA_PATH.resolve()}. Put higgs_subset.csv.gz (or higgs_subset.csv) beside the notebook '
        'or update DATA_PATH to its location.')
RESULTS_DIR = Path('results/higgs_notebook')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
N_THREADS = int(os.environ.get('SLURM_CPUS_PER_TASK', '1'))
print('CPU threads:', N_THREADS)

## 3. Read the CSV
The first column is `label`: **1 = signal**, **0 = background**. The other 28 columns are input features: the first 21 features (columns 2-22) are kinematic properties measured by the particle detectors in the accelerator. The last seven features are functions of the first 21 features; these are high-level features derived by physicists to help discriminate between the two classes. 

`pt` means transverse momentum, `eta` describes direction relative to the beam, and `phi` is an azimuthal angle. 

The CSV already has column names. Read it with a single `pd.read_csv` call. The CSV retains all observables for exploration. The basic-feature model uses the first 21 features; the derived-feature model uses the last seven. The label is excluded from both.

In [ ]:
FEATURES = [
    'lepton_pt', 'lepton_eta', 'lepton_phi',
    'missing_energy', 'missing_energy_phi',
    'jet1_pt', 'jet1_eta', 'jet1_phi', 'jet1_btag',
    'jet2_pt', 'jet2_eta', 'jet2_phi', 'jet2_btag',
    'jet3_pt', 'jet3_eta', 'jet3_phi', 'jet3_btag',
    'jet4_pt', 'jet4_eta', 'jet4_phi', 'jet4_btag',
    'm_jj', 'm_jjj', 'm_lv', 'm_jlv', 'm_bb', 'm_wbb', 'm_wwbb',
]
data = pd.read_csv(DATA_PATH)
source_path = DATA_PATH  # Used in the saved summary
print('Shape:', data.shape)
display(data.head())
display(data['label'].value_counts().rename(index={0: 'Background', 1: 'Signal'}))

## 4. Split the events once for both feature sets
Use 60% for training, 20% for selecting a score threshold, and 20% for the final test. Both models use exactly the same events in each split.

The validation sample is used only to choose a threshold retaining at least 80% of signal. Model settings are fixed in advance: 100 trees, depth 3, learning rate 0.1, without early stopping.

For 550,000 events this gives 330,000 training, 110,000 validation, and 110,000 test events. Test data is used for the final comparison after the thresholds are fixed.

In [ ]:
X = data[FEATURES]
y = data['label']
X_train, X_other, y_train, y_other = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=RANDOM_STATE)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_other, y_other, test_size=0.5, stratify=y_other, random_state=RANDOM_STATE)
assert 'label' not in X_train.columns
assert set(X_train.index).isdisjoint(X_valid.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_valid.index).isdisjoint(X_test.index)
print('Training / validation / test:', len(X_train), len(X_valid), len(X_test))

## Introduction: inspect transverse momentum
A single observable can be useful but may not fully separate the processes. First inspect the lepton transverse-momentum feature using training events.

The supplied variables are in their provided representation; axes below are not labeled in GeV. Histograms show the central 98% of pooled training values, with each displayed class normalized over that range.

In [ ]:
def plot_feature(feature):
    low, high = X_train[feature].quantile([0.01, 0.99])
    if high <= low:
        low, high = X_train[feature].min(), X_train[feature].max()
    if high <= low:
        print(f'{feature} is constant; choose another feature.')
        return
    bins = np.linspace(low, high, 41)
    fig, ax = plt.subplots()
    for label, name in [(0, 'Background'), (1, 'Signal')]:
        ax.hist(X_train.loc[y_train == label, feature], bins=bins,
                density=True, histtype='step', linewidth=2, label=name)
    ax.set(xlabel=f'{feature}', ylabel='Probability density',
           title='Training sample')
    ax.legend()
    fig.tight_layout()
    plt.show()

plot_feature('lepton_pt')

## Exercise 1 — Which observables distinguish the classes?
1. Plot `m_bb` for signal and background. This is the supplied two-b-jet mass feature; the dataset already contains its calculated value.
2. Choose two more features, such as `missing_energy`, `m_wwbb`, or `jet1_pt`, and plot them.

In [ ]:
# Try Exercise 1 here before running the worked example.
# Example: plot_feature('m_bb')

### Worked example for Exercise 1
Compare the signal/background overlap in these training-sample distributions. The two models below will combine either the basic observables or the derived masses.

In [ ]:
for feature in ['m_bb', 'missing_energy', 'm_wwbb']:
    plot_feature(feature)

## Select the 21 basic features
These are lepton kinematics, missing-energy variables, and four jets' kinematics/tagging. The label is excluded. Both models use 100 trees, maximum depth 3, and learning rate 0.1, with no early stopping.

In [ ]:
features1 = FEATURES[:21]
print('Basic features (21):', len(features1), 'features')
print(features1)

### Train XGBoost with basic features
Only training events are passed to `fit`. Higher signal scores mean more signal-like events.

In [ ]:
model1 = xgb.XGBClassifier(
    objective='binary:logistic',
    tree_method='hist',
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    eval_metric='logloss',
    n_jobs=N_THREADS,
    random_state=RANDOM_STATE,
)
start = perf_counter()
model1.fit(X_train[features1], y_train)
training_seconds1 = perf_counter() - start
print(f'Basic features (21): training finished in {training_seconds1:.2f} seconds')

### Set the selection threshold for basic features
Use validation scores to find the strictest threshold that retains at least **80% of validation signal**. This selection happens after training and does not change the fitted model. 

In [ ]:
TARGET_SIGNAL_EFFICIENCY = 0.80
valid_scores1 = model1.predict_proba(X_valid[features1])[:, 1]
fpr, tpr, cuts = roc_curve(y_valid, valid_scores1)
eligible = np.flatnonzero(np.isfinite(cuts) & (tpr >= TARGET_SIGNAL_EFFICIENCY))
threshold1 = float(cuts[eligible[0]])

### Print final test results: basic features
Signal efficiency is the fraction of signal retained; background efficiency is the fraction of background retained. Lower background efficiency is better at comparable signal efficiency. Background rejection is its reciprocal. Test signal efficiency can differ slightly from the validation target. ROC AUC summarizes ranking across thresholds.

In [ ]:
test_labels = y_test.to_numpy()
scores1 = model1.predict_proba(X_test[features1])[:, 1]
accepted1 = scores1 >= threshold1
signal_eff1 = float(accepted1[test_labels == 1].mean())
background_eff1 = float(accepted1[test_labels == 0].mean())
result1 = {
    'feature_set': 'Basic features (21)',
    'input_features': len(features1),
    'training_seconds': training_seconds1,
    'test_auc': float(roc_auc_score(test_labels, scores1)),
    'threshold': threshold1,
    'test_signal_efficiency': signal_eff1,
    'test_background_efficiency': background_eff1,
    'test_background_rejection': 1 / background_eff1 if background_eff1 > 0 else np.nan,
}
display(pd.DataFrame([result1]).set_index('feature_set').round(4))

### Plot final test results: basic features
Show signal scores, the ROC curve, and background rejection. The dashed vertical line on the score plot is the frozen threshold. Zero observed background acceptance is omitted from the logarithmic rejection plot; a finite sample does not establish infinite rejection.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for label, label_name in [(0, 'Background'), (1, 'Signal')]:
    axes[0].hist(scores1[test_labels == label], bins=np.linspace(0, 1, 31),
                 density=True, histtype='step', linewidth=2, label=label_name)
axes[0].axvline(threshold1, color='black', linestyle='--', label='Chosen threshold')
axes[0].set(xlabel='XGBoost signal score', ylabel='Probability density')
fpr, tpr, _ = roc_curve(y_test, scores1)
axes[1].plot(fpr, tpr, label=f"AUC={result1['test_auc']:.3f}")
axes[1].plot([0, 1], [0, 1], '--', color='gray', label='Random ranking')
axes[1].plot(background_eff1, signal_eff1, 'o', color='black', label='Frozen threshold')
axes[1].set(xlabel='Background efficiency', ylabel='Signal efficiency',
            xlim=(0, 1), ylim=(0, 1))
visible = (fpr > 0) & (tpr > 0)
axes[2].plot(tpr[visible], 1 / fpr[visible], label='Basic features (21)')
random_eff = np.linspace(0.01, 1, 300)
axes[2].plot(random_eff, 1 / random_eff, '--', color='gray', label='Random ranking')
if background_eff1 > 0:
    axes[2].plot(signal_eff1, 1 / background_eff1, 'o', color='black', label='Frozen threshold')
axes[2].set(xlabel='Signal efficiency', ylabel='Background rejection (1/efficiency)',
            xlim=(0, 1), yscale='log')
for ax in axes:
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)
fig.suptitle('Basic features (21): final test sample')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'basic_features_test_results.png', dpi=150)
plt.show()

### Save results for basic features

In [ ]:
model1.save_model(RESULTS_DIR / 'basic_features_model.json')
pd.DataFrame({'label': test_labels, 'score': scores1}).to_csv(
    RESULTS_DIR / 'basic_features_test_predictions.csv', index=False)
with open(RESULTS_DIR / 'basic_features_summary.json', 'w') as handle:
    json.dump(result1, handle, indent=2)
print('Basic-feature outputs saved to:', RESULTS_DIR.resolve())

# Stage 2: Train with seven physics-derived features

Continue here in the **same Jupyter Notebook and kernel**, after completing Stage 1. The seven inputs are `m_jj`, `m_jjj`, `m_lv`, `m_jlv`, `m_bb`, `m_wbb`, and `m_wwbb`.

The following cells repeat the training, validation-threshold selection, test evaluation, and plotting workflow. Only the feature group changes. Reuse the existing event splits and fixed model settings.

You do not need to reload the CSV, recreate the splits, or restart the kernel between stages. If you have restarted the kernel, rerun the notebook from the beginning so that both models and their results exist in memory.

In [ ]:
features2 = FEATURES[21:]
print('Derived features (7):', len(features2), 'features')
print(features2)

### Train XGBoost with physics-derived features
Only training events are passed to `fit`. Higher signal scores mean more signal-like events.

In [ ]:
model2 = xgb.XGBClassifier(
    objective='binary:logistic',
    tree_method='hist',
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    eval_metric='logloss',
    n_jobs=N_THREADS,
    random_state=RANDOM_STATE,
)
start = perf_counter()
model2.fit(X_train[features2], y_train)
training_seconds2 = perf_counter() - start
print(f'Derived features (7): training finished in {training_seconds2:.2f} seconds')

### Set the selection threshold for derived features
Use validation scores to find the strictest threshold retaining at least **80% of validation signal**. This selection happens after training and does not change the fitted model. No validation results are plotted. Freeze the threshold before evaluating test events.

In [ ]:
TARGET_SIGNAL_EFFICIENCY = 0.80
valid_scores2 = model2.predict_proba(X_valid[features2])[:, 1]
fpr, tpr, cuts = roc_curve(y_valid, valid_scores2)
eligible = np.flatnonzero(np.isfinite(cuts) & (tpr >= TARGET_SIGNAL_EFFICIENCY))
threshold2 = float(cuts[eligible[0]])

### Print final test results: derived features
Signal efficiency is the fraction of signal retained; background efficiency is the fraction of background retained. Lower background efficiency is better at comparable signal efficiency. Background rejection is its reciprocal. Test signal efficiency can differ slightly from the validation target. ROC AUC summarizes ranking across thresholds.

In [ ]:
test_labels = y_test.to_numpy()
scores2 = model2.predict_proba(X_test[features2])[:, 1]
accepted2 = scores2 >= threshold2
signal_eff2 = float(accepted2[test_labels == 1].mean())
background_eff2 = float(accepted2[test_labels == 0].mean())
result2 = {
    'feature_set': 'Derived features (7)',
    'input_features': len(features2),
    'training_seconds': training_seconds2,
    'test_auc': float(roc_auc_score(test_labels, scores2)),
    'threshold': threshold2,
    'test_signal_efficiency': signal_eff2,
    'test_background_efficiency': background_eff2,
    'test_background_rejection': 1 / background_eff2 if background_eff2 > 0 else np.nan,
}
display(pd.DataFrame([result2]).set_index('feature_set').round(4))

### Plot final test results: derived features
Show signal scores, the ROC curve, and background rejection. The dashed vertical line on the score plot is the frozen threshold.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for label, label_name in [(0, 'Background'), (1, 'Signal')]:
    axes[0].hist(scores2[test_labels == label], bins=np.linspace(0, 1, 31),
                 density=True, histtype='step', linewidth=2, label=label_name)
axes[0].axvline(threshold2, color='black', linestyle='--', label='Chosen threshold')
axes[0].set(xlabel='XGBoost signal score', ylabel='Probability density')
fpr, tpr, _ = roc_curve(y_test, scores2)
axes[1].plot(fpr, tpr, label=f"AUC={result2['test_auc']:.3f}")
axes[1].plot([0, 1], [0, 1], '--', color='gray', label='Random ranking')
axes[1].plot(background_eff2, signal_eff2, 'o', color='black', label='Frozen threshold')
axes[1].set(xlabel='Background efficiency', ylabel='Signal efficiency',
            xlim=(0, 1), ylim=(0, 1))
visible = (fpr > 0) & (tpr > 0)
axes[2].plot(tpr[visible], 1 / fpr[visible], label='Derived features (7)')
random_eff = np.linspace(0.01, 1, 300)
axes[2].plot(random_eff, 1 / random_eff, '--', color='gray', label='Random ranking')
if background_eff2 > 0:
    axes[2].plot(signal_eff2, 1 / background_eff2, 'o', color='black', label='Frozen threshold')
axes[2].set(xlabel='Signal efficiency', ylabel='Background rejection (1/efficiency)',
            xlim=(0, 1), yscale='log')
for ax in axes:
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)
fig.suptitle('Derived features (7): final test sample')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'derived_features_test_results.png', dpi=150)
plt.show()

### Save results for derived features

In [ ]:
model2.save_model(RESULTS_DIR / 'derived_features_model.json')
pd.DataFrame({'label': test_labels, 'score': scores2}).to_csv(
    RESULTS_DIR / 'derived_features_test_predictions.csv', index=False)
with open(RESULTS_DIR / 'derived_features_summary.json', 'w') as handle:
    json.dump(result2, handle, indent=2)
print('Derived-feature outputs saved to:', RESULTS_DIR.resolve())

## Compare basic and physics-derived features

Compare the two models trained in this Jupyter session. Both thresholds were selected on validation data at the same 80% target signal efficiency, then frozen before final testing.

The next cell uses the in-memory results and predictions from both stages. Compare background efficiency and rejection, ROC AUC, and training time on the same test events. Do not retune against these test results. The notebook also saves the comparison table and plots for later use.

In [ ]:
# Compare the two models from this notebook run.
comparison = pd.DataFrame([result1, result2]).set_index('feature_set')
display(comparison.round(4))
comparison.to_csv(RESULTS_DIR / 'feature_comparison.csv')

test_scores = {
    'Basic features (21)': scores1,
    'Derived features (7)': scores2,
}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name, scores in test_scores.items():
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = comparison.loc[name, 'test_auc']
    color = axes[0].plot(fpr, tpr, label=f'{name}: AUC={auc:.3f}')[0].get_color()
    visible = (fpr > 0) & (tpr > 0)
    axes[1].plot(tpr[visible], 1 / fpr[visible], color=color, label=name)
    # Mark efficiencies at the threshold frozen before test evaluation.
    signal_eff = comparison.loc[name, 'test_signal_efficiency']
    background_eff = comparison.loc[name, 'test_background_efficiency']
    axes[0].plot(background_eff, signal_eff, 'o', color=color)
    if background_eff > 0:
        axes[1].plot(signal_eff, 1 / background_eff, 'o', color=color)
axes[0].plot([0, 1], [0, 1], '--', color='gray', label='Random ranking')
random_eff = np.linspace(0.01, 1, 300)
axes[1].plot(random_eff, 1 / random_eff, '--', color='gray', label='Random ranking')
axes[0].set(xlabel='Background efficiency', ylabel='Signal efficiency',
            xlim=(0, 1), ylim=(0, 1))
axes[1].set(xlabel='Signal efficiency', ylabel='Background rejection (1/efficiency)',
            xlim=(0, 1), yscale='log')
for ax in axes:
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)
fig.suptitle('Final test comparison (dots: frozen thresholds)')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'test_feature_comparison.png', dpi=150)
plt.show()

In [ ]:
settings = {
    'data_path': str(DATA_PATH), 'events': len(data),
    'random_state': RANDOM_STATE, 'xgboost_version': xgb.__version__,
    'features': {'Basic features (21)': FEATURES[:21], 'Derived features (7)': FEATURES[21:]},
    'thresholds': {name: float(comparison.loc[name, 'threshold']) for name in comparison.index},
    'target_validation_signal_efficiency': TARGET_SIGNAL_EFFICIENCY,
    'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1,
}
with open(RESULTS_DIR / 'settings.json', 'w') as handle:
    json.dump(settings, handle, indent=2)
print('Results saved to:', RESULTS_DIR.resolve())

## Discussion
Does XGBoost distinguish signal from background better using basic features or physics-derived features? How do the ROC curves and training times compare?

## References
- Whiteson, D. (2014). *HIGGS [Dataset]*. [UCI Machine Learning Repository](https://doi.org/10.24432/C5V312). CC BY 4.0.
- Baldi, P., Sadowski, P., and Whiteson, D. (2014). [Searching for exotic particles in high-energy physics with deep learning](https://doi.org/10.1038/ncomms5308).
- [XGBoost documentation](https://xgboost.readthedocs.io/).